**Objetivo**

Promover as metas nacionais de alfabetização da camada Bronze para a Silver, padronizando tipos, rede de ensino e campos de auditoria.

**Fonte de dados**

- `bronze.meta_alfabetizacao_brasil`

**Destino**

- `silver.meta_alfabetizacao_brasil`

**Granularidade**

- Uma linha por `ano` e `rede`.

> As validações de qualidade são informativas, como no notebook de município. Apenas a ausência de colunas obrigatórias impede tecnicamente a execução.

## 0. Configurando sessão Spark

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("bronze_to_silver_meta_alfabetizacao_brasil")
    .config(
        "spark.jars.packages",
        "com.google.cloud.spark:spark-bigquery-with-dependencies_2.13:0.44.2"
    )
    .getOrCreate()
)

spark.conf.set("parentProject", "tech-challenge-fase-2-505123")

:: loading settings :: url = jar:file:/opt/micromamba/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/jupyter/.ivy2.5.2/cache
The jars for the packages stored in: /home/jupyter/.ivy2.5.2/jars
com.google.cloud.spark#spark-bigquery-with-dependencies_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-aec6e475-718a-4876-8b52-1a9c8c6636aa;1.0
	confs: [default]
	found com.google.cloud.spark#spark-bigquery-with-dependencies_2.13;0.44.2 in central
:: resolution report :: resolve 191ms :: artifacts dl 4ms
	:: modules in use:
	com.google.cloud.spark#spark-bigquery-with-dependencies_2.13;0.44.2 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-----------------------------------------

In [2]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)
spark.conf.set("spark.sql.repl.eagerEval.maxNumRows", 20)
spark.conf.set("spark.sql.repl.eagerEval.truncate", 100)

## 1. Imports

In [3]:
from pyspark.sql import functions as F

## 2. Geração de parâmetros

In [4]:
par_source_project = "tech-challenge-fase-2-505123"
par_source_bronze_meta = f"{par_source_project}.bronze.meta_alfabetizacao_brasil"
par_source_silver_meta = f"{par_source_project}.silver.meta_alfabetizacao_brasil"

colunas_meta = [f"meta_alfabetizacao_{ano}" for ano in range(2024, 2031)]
colunas_percentuais = ["taxa_alfabetizacao", *colunas_meta, "percentual_participacao"]
colunas_esperadas = [
    "ano", "rede", *colunas_percentuais, "_ingestao_timestamp", "_fonte"
]

## 3. Leitura dos dados da origem

In [5]:
df_src_meta = (
    spark.read.format("bigquery")
    .option("table", par_source_bronze_meta)
    .load()
)

### 3.1. Validação do contrato de entrada

In [6]:
colunas_ausentes = sorted(set(colunas_esperadas) - set(df_src_meta.columns))
if colunas_ausentes:
    raise ValueError(f"Schema inválido. Colunas ausentes na Bronze: {colunas_ausentes}")

df_meta = df_src_meta.select(*colunas_esperadas)

### 3.2. Diagnóstico do domínio de rede

In [7]:
(
    df_meta.groupBy("rede").count()
    .orderBy(F.desc("count"), F.asc_nulls_first("rede"))
    .show(100, truncate=False)
)

[Stage 0:===========================================================(1 + 0) / 1]

+-------+-----+
|rede   |count|
+-------+-----+
|Pública|3    |
+-------+-----+



## 4. Transformações

In [8]:
rede_limpa = F.trim(F.col("rede").cast("string"))
rede_minuscula = F.lower(rede_limpa)
variantes_rede_publica = ["pública", "publica", "p�blica", "pãºblica"]

df_silver_meta = (
    df_meta
    .withColumn("ano", F.col("ano").cast("int"))
    .withColumn(
        "rede",
        F.when(rede_minuscula.isin(*variantes_rede_publica), F.lit("Pública"))
        .when(rede_limpa == "", F.lit(None))
        .otherwise(F.initcap(rede_minuscula))
    )
    .withColumn("_ingestao_timestamp", F.col("_ingestao_timestamp").cast("timestamp"))
    .withColumn("_fonte", F.trim(F.col("_fonte")))
)

for coluna in colunas_percentuais:
    df_silver_meta = df_silver_meta.withColumn(coluna, F.col(coluna).cast("double"))

### 4.1. Data de carregamento e exclusão de duplicadas

In [9]:
chave = ["ano", "rede"]
df_silver_meta_antes_dedup = df_silver_meta
df_silver_meta = (
    df_silver_meta
    .dropDuplicates(chave)
    .withColumn("_silver_timestamp", F.current_timestamp())
)

## 5. Validação da qualidade

In [10]:
print("=== Relatório de Qualidade — silver.meta_alfabetizacao_brasil ===")

qtd_bronze = df_src_meta.count()
qtd_silver = df_silver_meta.count()

# 1) Duplicidade na chave natural
dups_antes = (
    df_silver_meta_antes_dedup.groupBy(*chave).count()
    .filter(F.col("count") > 1).count()
)
dups_depois = (
    df_silver_meta.groupBy(*chave).count()
    .filter(F.col("count") > 1).count()
)
print(f"Chaves duplicadas antes da deduplicação: {dups_antes}")
print(f"Chaves duplicadas após deduplicação: {dups_depois}")

# 2) Domínio de rede esperado para esta tabela nacional
print("Redes diferentes de 'Pública':")
(
    df_silver_meta.filter(F.col("rede").isNotNull() & (F.col("rede") != "Pública"))
    .groupBy("rede").count().orderBy(F.desc("count"))
    .show(100, truncate=False)
)

# 3) Nulos nas colunas críticas
for coluna in colunas_esperadas:
    quantidade = df_silver_meta.filter(F.col(coluna).isNull()).count()
    print(f"Nulos em '{coluna}': {quantidade}")

# 4) Percentuais fora da faixa esperada [0, 100]
for coluna in colunas_percentuais:
    quantidade = df_silver_meta.filter(
        (F.col(coluna) < 0) | (F.col(coluna) > 100) | F.isnan(coluna)
    ).count()
    print(f"Valores fora de [0, 100] ou NaN em '{coluna}': {quantidade}")

# 5) As metas anuais devem permanecer iguais ou crescer ao longo do horizonte
meta_nao_monotona = df_silver_meta.filter(
    (F.col("meta_alfabetizacao_2025") < F.col("meta_alfabetizacao_2024"))
    | (F.col("meta_alfabetizacao_2026") < F.col("meta_alfabetizacao_2025"))
    | (F.col("meta_alfabetizacao_2027") < F.col("meta_alfabetizacao_2026"))
    | (F.col("meta_alfabetizacao_2028") < F.col("meta_alfabetizacao_2027"))
    | (F.col("meta_alfabetizacao_2029") < F.col("meta_alfabetizacao_2028"))
    | (F.col("meta_alfabetizacao_2030") < F.col("meta_alfabetizacao_2029"))
).count()
print(f"Linhas com metas decrescentes: {meta_nao_monotona}")

print(f"Linhas Bronze: {qtd_bronze} -> Linhas Silver: {qtd_silver}")

=== Relatório de Qualidade — silver.meta_alfabetizacao_brasil ===


Chaves duplicadas antes da deduplicação: 0
Chaves duplicadas após deduplicação: 0
Redes diferentes de 'Pública':
+----+-----+
|rede|count|
+----+-----+
+----+-----+

Nulos em 'ano': 0
Nulos em 'rede': 0
Nulos em 'taxa_alfabetizacao': 0
Nulos em 'meta_alfabetizacao_2024': 0
Nulos em 'meta_alfabetizacao_2025': 0
Nulos em 'meta_alfabetizacao_2026': 0
Nulos em 'meta_alfabetizacao_2027': 0
Nulos em 'meta_alfabetizacao_2028': 0
Nulos em 'meta_alfabetizacao_2029': 0
Nulos em 'meta_alfabetizacao_2030': 0
Nulos em 'percentual_participacao': 0
Nulos em '_ingestao_timestamp': 0
Nulos em '_fonte': 0
Valores fora de [0, 100] ou NaN em 'taxa_alfabetizacao': 0
Valores fora de [0, 100] ou NaN em 'meta_alfabetizacao_2024': 0
Valores fora de [0, 100] ou NaN em 'meta_alfabetizacao_2025': 0
Valores fora de [0, 100] ou NaN em 'meta_alfabetizacao_2026': 0
Valores fora de [0, 100] ou NaN em 'meta_alfabetizacao_2027': 0
Valores fora de [0, 100] ou NaN em 'meta_alfabetizacao_2028': 0
Valores fora de [0, 100] o

## 6. Armazenamento no BigQuery

In [11]:
(
    df_silver_meta.write.format("bigquery")
    .option("table", par_source_silver_meta)
    .option("writeMethod", "direct")
    .option("clusteredFields", "ano,rede")
    .mode("overwrite")
    .save()
)

26/08/24 01:37:23 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                